# SituationCatch-Bench — 인간 IAA 핸드오프 노트북 (Kaggle / Colab / Local)

**목적.** 서로 독립적인 **사람 주석자 3명**이 같은 70개 문항의 *상황 상태(situation state)* 를
각자 판단해 채우면, 이 노트북이 저장소의 실제 CLI(`code/annotation_cli.py`)를 그대로 호출하여
**Fleiss κ(슬롯별 일치도)** 와 **불일치표(adjudication)** 를 계산하고 결과를 ZIP으로 내보냅니다.

이 결과는 논문 *"Situation engineering"* 의 리뷰어 요구사항 **M-B(인간 검증)** 를
"준비됨"에서 **"실측 완료"** 로 바꾸기 위한 것입니다.

> ⚠️ **Kaggle 사용자 필독:** 오른쪽 패널 **Settings → Internet → On** 을 켜세요
> (STEP 0에서 GitHub 저장소를 clone 하려면 인터넷이 필요합니다).
> pip 설치는 필요 없습니다 — 채점 코드는 파이썬 표준 라이브러리만 사용합니다.

> 🔬 **연구윤리.** STEP 3의 *시뮬레이션 페르소나* 는 파이프라인 점검용 **합성 데이터**이며,
> `NOT HUMAN-SUBJECT EVIDENCE` 로 명시됩니다 — 사람 데이터가 **아닙니다**.
> 논문에 반영되는 실측 수치는 STEP 4~5의 **사람이 채운 packet** 에서만 나옵니다.

---
### 실행 순서 요약
| STEP | 내용 | 사람 개입 |
|------|------|-----------|
| 0 | 환경 감지 + 저장소 clone | 자동 |
| 1 | CLI 동작 확인 | 자동 |
| 2 | 작업 폴더 준비(빈 packet + 예시 + 코드북 복사) | 자동 |
| 3 | ⭐ 시뮬레이션 페르소나로 파이프라인 검증 (합성, 논문 미반영) | 자동 |
| 4 | ⭐ **사람이 채운 packet** 채점 → `agreement.json` | **사람 필요** |
| 5 | 불일치표 생성 → `adjudication.csv` | 사람 필요 |
| 6 | 결과 ZIP 내보내기 | 자동 |

## STEP 0 — 환경 감지 + 저장소 준비

Kaggle(`/kaggle/working`) / Colab(`/content`) / 로컬을 자동 감지하고,
`leemgs/sage` 저장소를 clone 합니다 (이미 있으면 건너뜀). 경로 수동 설정 불필요.

In [ ]:
import os, sys, subprocess, shutil, glob, json, textwrap
from pathlib import Path

REPO_URL = 'https://github.com/leemgs/sage.git'

def detect_base():
    if Path('/kaggle/working').exists() or os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        return Path('/kaggle/working'), 'kaggle'
    try:
        import google.colab  # noqa: F401
        return Path('/content'), 'colab'
    except Exception:
        return Path.cwd(), 'local'

BASE, ENV = detect_base()
BASE.mkdir(parents=True, exist_ok=True)
print(f'[env] detected = {ENV}')
print(f'[env] BASE     = {BASE}')

REPO = BASE / 'sage'
if (REPO / 'code' / 'annotation_cli.py').exists():
    print(f'[repo] already present at {REPO}')
else:
    print(f'[repo] cloning {REPO_URL} -> {REPO} ...')
    r = subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)
    if not (REPO / 'code' / 'annotation_cli.py').exists():
        raise SystemExit('[repo] clone failed. On Kaggle enable Settings -> Internet -> On, '
                         'then Run All again.')
    print('[repo] clone OK')

CLI      = REPO / 'code' / 'annotation_cli.py'
PACKETS  = REPO / 'paper' / 'annotation_packets'
SUBSET   = REPO / 'paper' / 'data' / 'annotation_subset_70.jsonl'
print('[repo] CLI     =', CLI)
print('[repo] PACKETS =', PACKETS)

## STEP 1 — CLI 동작 확인

채점 코드가 정상적으로 로드되는지 `--help` 로 확인합니다 (표준 라이브러리만 사용).

In [ ]:
r = subprocess.run([sys.executable, str(CLI), '--help'], capture_output=True, text=True)
print(r.stdout or r.stderr)
assert 'score' in (r.stdout + r.stderr), 'annotation_cli.py did not load as expected'
print('[ok] CLI loaded — subcommands: init / score / simulate-personas / adjudicate')

## STEP 2 — 작업 폴더 준비

`human_iaa_work/` 작업 폴더를 만들고, 저장소의 **빈 packet 3개**, **정답 예시**,
**코드북/작성 사례집**을 복사합니다. 사람이 채워야 할 파일이 바로 여기에 놓입니다.

이미 채운 CSV가 있다면 아래 두 방법 중 하나로 넣으세요:
- **Kaggle:** 오른쪽 **+ Add Input → Upload** 로 `annotator_1/2/3.csv` 업로드
  → STEP 4가 `/kaggle/input` 을 자동 탐색해 가져옵니다.
- **Colab/로컬:** 왼쪽 파일 탭에서 `human_iaa_work/annotator_1/2/3.csv` 를 직접 덮어쓰기.

In [ ]:
WORK = BASE / 'human_iaa_work'
WORK.mkdir(parents=True, exist_ok=True)

COPY = ['annotator_1.csv', 'annotator_2.csv', 'annotator_3.csv',
        'EXAMPLE_annotator_filled.csv', 'HOW_TO_ANNOTATE.md', 'EXAMPLE_WALKTHROUGH.md']
for name in COPY:
    src = PACKETS / name
    if src.exists():
        shutil.copy2(src, WORK / name)
        print('[copy]', name)
    else:
        print('[warn] missing in repo:', name)

print()
print('작업 폴더:', WORK)
print('사람이 채울 파일 : annotator_1.csv / annotator_2.csv / annotator_3.csv')
print('참고용(수정 금지): EXAMPLE_annotator_filled.csv, HOW_TO_ANNOTATE.md, EXAMPLE_WALKTHROUGH.md')

## STEP 3 — ⭐ 파이프라인 검증 (시뮬레이션 페르소나 · 합성 · 논문 미반영)

사람이 시간을 들이기 전에, **채점 파이프라인이 이 환경에서 정상 동작하는지** 먼저 확인합니다.
저장소의 `simulate-personas` 로 70문항에 대한 **결정론적 합성 라벨 3벌**을 만들고
score/adjudicate 를 돌려 봅니다.

> ⚠️ 여기서 나오는 κ 값은 **합성 데이터** 결과입니다. `SIMULATION_MANIFEST.json` 에
> `NOT HUMAN-SUBJECT EVIDENCE` 로 표시되며 **논문에 들어가지 않습니다.**
> (이 셀은 건너뛰어도 STEP 4~5에는 영향 없음.)

In [ ]:
SIM = WORK / '_sim_validation'
if SIM.exists():
    shutil.rmtree(SIM)
r = subprocess.run([sys.executable, str(CLI), 'simulate-personas',
                    '--input', str(SUBSET), '--out', str(SIM),
                    '--seed', '7', '--disagreement-rate', '0.15'],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)

r = subprocess.run([sys.executable, str(CLI), 'score',
                    '--annotations', str(SIM), '--out', str(SIM / 'agreement.json')],
                   capture_output=True, text=True)
if r.returncode == 0:
    rep = json.loads((SIM / 'agreement.json').read_text())
    print('provenance :', rep['provenance'], '| synthetic:', rep['synthetic'])
    print('warning    :', rep.get('warning', '(none)'))
    print('슬롯별 Fleiss κ (합성):')
    for slot, v in rep['slots'].items():
        print(f"  {slot:<15} kappa={v['fleiss_kappa']:.3f}  unanimous={v['unanimous_rate']:.2f}")
    subprocess.run([sys.executable, str(CLI), 'adjudicate',
                    '--annotations', str(SIM), '--out', str(SIM / 'adjudication.csv')])
    print('\n[ok] 파이프라인 검증 성공 — score/adjudicate 정상 동작합니다.')
else:
    print(r.stderr)
    print('[warn] 검증 실패 — STEP 0~2 출력을 확인하세요.')

## STEP 4 — ⭐ 사람이 채운 packet 채점 (실측 · 논문 반영)

**이 단계가 실제 결과입니다.** `annotator_1/2/3.csv` 의 9개 열(코드북 참고)이
모두 채워져 있어야 합니다. 코드는 fail-closed 로 동작하여, 빈 칸/누락/개수 부족 시
명확한 오류를 내고 채점을 거부합니다 (무결성 보장).

- Kaggle: `/kaggle/input/**/annotator_*.csv` 를 자동 탐색해 작업 폴더로 가져옵니다
  (`EXAMPLE_` 파일은 제외).
- 아직 안 채웠다면 이 셀은 **안내만 출력하고 건너뜁니다.** 채운 뒤 다시 **Run All** 하세요.

In [ ]:
import csv
RATING_SLOTS = ['action','answer','temporal_state','modality','scope',
                'source_status','observer_state','world']

# (Kaggle) 업로드된 데이터셋(/kaggle/input)에서 채운 packet 자동 수집.
# 무결성 가드: (1) 합성 데이터(SIMULATION_MANIFEST.json 이 있는 폴더)는 절대 가져오지 않음,
# (2) EXAMPLE_ 파일 제외, (3) 작업 폴더(WORK) 자체와 그 하위(_sim_validation)는 스캔 안 함.
picked = {}
for p in glob.glob('/kaggle/input/**/annotator_*.csv', recursive=True):
    pp = Path(p)
    if pp.name.startswith('EXAMPLE_'):
        continue
    if (pp.parent / 'SIMULATION_MANIFEST.json').exists():
        print('[skip] 합성 데이터로 판단되어 제외:', p)
        continue
    picked[pp.name] = p  # 같은 파일명은 마지막 것 사용
for name, p in sorted(picked.items()):
    shutil.copy2(p, WORK / name)
    print('[import] 업로드본 반영:', name, '<-', p)

def is_filled(path):
    try:
        rows = list(csv.DictReader(open(path, encoding='utf-8')))
    except Exception:
        return False
    return bool(rows) and all(str(r.get(s, '')).strip() for r in rows for s in RATING_SLOTS)

human = sorted(WORK.glob('annotator_*.csv'))
human = [p for p in human if not p.name.startswith('EXAMPLE_')]
filled = [p for p in human if is_filled(p)]
print(f'\n[status] packet 파일 {len(human)}개 중 채워진 것 {len(filled)}개')

AGREE = WORK / 'agreement.json'
ADJ_stale = WORK / 'adjudication.csv'
for stale in (AGREE, ADJ_stale):
    if stale.exists():
        stale.unlink()  # 이전 실행 결과가 이번 실행을 오염시키지 않도록 제거
if len(filled) >= 3:
    r = subprocess.run([sys.executable, str(CLI), 'score',
                        '--annotations', str(WORK), '--out', str(AGREE),
                        '--provenance', 'human_annotations'],
                       capture_output=True, text=True)
    if r.returncode == 0:
        rep = json.loads(AGREE.read_text())
        print('[ok] 실측 채점 완료 — provenance =', rep['provenance'])
        print(f"annotators={rep['n_annotators']}  items={rep['n_items']}")
        print('슬롯별 Fleiss κ (실측):')
        for slot, v in rep['slots'].items():
            print(f"  {slot:<15} kappa={v['fleiss_kappa']:.3f}  unanimous={v['unanimous_rate']:.2f}")
    else:
        print('[fail-closed] 채점 거부:')
        print(r.stderr.strip())
        print('\n위 메시지가 가리키는 파일/열을 채운 뒤 다시 Run All 하세요.')
else:
    print(textwrap.dedent('''
        [대기] 아직 사람 주석이 완료되지 않았습니다.

        해야 할 일:
          1) 독립적인 주석자 3명에게 annotator_1.csv / _2.csv / _3.csv 를 각각 배부
          2) 코드북(HOW_TO_ANNOTATE.md)과 작성 사례집(EXAMPLE_WALKTHROUGH.md) 참고하여
             9개 열(action, answer, temporal_state, modality, scope,
             source_status, observer_state, world)을 모두 채움 (notes는 선택)
          3) blind_id/item_id/question/evidence 4개 열은 절대 수정 금지
          4) 채운 3개 CSV를 작업 폴더에 넣고(또는 Kaggle Add Input 업로드) 다시 Run All
    '''))

## STEP 5 — 불일치표(adjudication) 생성

주석자 간 라벨이 갈린 (item, slot) 조합을 뽑아 `adjudication.csv` 로 저장합니다.
빈 `adjudicated_label` / `adjudicator_rationale` 열은 최종 판정을 적어 넣는 칸입니다.
(STEP 4가 아직 대기 상태면 이 셀도 자동으로 건너뜁니다.)

In [ ]:
ADJ = WORK / 'adjudication.csv'
if AGREE.exists():
    r = subprocess.run([sys.executable, str(CLI), 'adjudicate',
                        '--annotations', str(WORK), '--out', str(ADJ)],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
    if ADJ.exists():
        rows = list(csv.DictReader(open(ADJ, encoding='utf-8')))
        print(f'[ok] 불일치 {len(rows)}건 -> {ADJ}')
        for row in rows[:5]:
            print(f"  {row['item_id']:<20} {row['slot']:<14} {row['independent_labels']}")
        if len(rows) > 5:
            print(f'  ... (+{len(rows)-5} more)')
else:
    print('[skip] STEP 4의 실측 채점이 끝난 뒤 실행됩니다.')

## STEP 6 — 결과 ZIP 내보내기

`agreement.json` + `adjudication.csv` + 채운 packet 3개를 `human_iaa_results.zip` 으로 묶습니다.

- **Kaggle:** 오른쪽 **Output** 패널에서 다운로드 (`/kaggle/working/human_iaa_results.zip`).
- **Colab:** 자동 다운로드 시도.

이 ZIP(또는 `agreement.json`)을 연구자에게 전달하면 논문 Methods/Results/Evidence-ladder 와
`RESPONSE_TO_REVIEWERS.md` 의 **M-B** 항목에 반영됩니다.

In [ ]:
import zipfile
ZIP = BASE / 'human_iaa_results.zip'
members = []
for name in ['agreement.json', 'adjudication.csv',
             'annotator_1.csv', 'annotator_2.csv', 'annotator_3.csv']:
    p = WORK / name
    if p.exists():
        members.append(p)

if not (WORK / 'agreement.json').exists():
    print('[note] 실측 결과(agreement.json)가 아직 없습니다 — 채운 packet만 담아 백업 ZIP을 만듭니다.')

with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in members:
        z.write(p, arcname=p.name)
print('[zip]', ZIP, '\n포함:', [p.name for p in members] or '(없음)')

if ENV == 'colab':
    try:
        from google.colab import files
        files.download(str(ZIP))
    except Exception as e:
        print('[colab] 자동 다운로드 실패 — 왼쪽 파일 탭에서 직접 받으세요:', e)
elif ENV == 'kaggle':
    print('[kaggle] 오른쪽 Output 패널에서 human_iaa_results.zip 을 다운로드하세요.')
else:
    print('[local] 저장 위치:', ZIP)

---
### 마무리 체크리스트
- [ ] STEP 3 (합성) 이 초록불(κ 출력) → 파이프라인 정상
- [ ] 주석자 3명이 독립적으로 packet 9개 열 작성 완료
- [ ] STEP 4 가 `provenance = human_annotations` 로 실측 κ 출력
- [ ] STEP 6 의 `human_iaa_results.zip` 을 연구자에게 전달

문의/코드북: `human_iaa_work/HOW_TO_ANNOTATE.md`, 작성 사례: `human_iaa_work/EXAMPLE_WALKTHROUGH.md`